# Une installation face au prix : un électrolyseur, deux chaudières

*Le squelette du premier modèle appliqué à deux installations qui achètent l'électricité au prix horaire français de 2030 : un électrolyseur avec une cavité saline, qui tourne 63 % du temps et paie son électricité 36,18 €/MWh au lieu de 57,50, puis une chaudière électrique à côté d'une chaudière à gaz, qui tourne exactement les 3 680 heures sous le prix de bascule de 44,54 €/MWh.*

Ce notebook est tiré du texte [en ligne](https://www.robingirard.eu/assets/parcours-es/latest/textes/installation-face-au-prix.html) du parcours ES ; exécutez les cellules dans l'ordre (Maj + Entrée). Les commentaires des cellules donnent les valeurs attendues.

**La cellule d'installation.** En local, placez ce notebook dans le dépôt `pommes-case-studies` (à sa racine ou dans un de ses dossiers) et ouvrez-le avec le noyau du venv du dépôt (`.venv`) : la cellule n'installe rien, elle trouve le dépôt et affiche les versions des paquets.

Sur Google Colab (Fichier › Importer un notebook), la cellule clone le dépôt et installe les paquets : 2 à 3 minutes, puis elle s'arrête volontairement sur `RedemarrezLaSession`. Faites Exécution › Redémarrer la session, relancez cette cellule, et seulement elle, puis continuez dans l'ordre.

In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

VERSIONS = {"pommes": "0.11.0", "pommes-craft": "0.6.0", "linopy": "0.10.1", "highspy": "1.15.1"}


def versions_du_depot() -> bool:
    """Vrai si les paquets installés sont ceux de requirements.txt (les valeurs attendues en dépendent)."""
    try:
        return all(version(nom) == v for nom, v in VERSIONS.items())
    except PackageNotFoundError:
        return False


class RedemarrezLaSession(Exception):
    """Colab : les paquets sont installés, mais la session garde en mémoire les anciens numpy et pandas."""


if "google.colab" in sys.modules:                     # sur Colab : cloner, installer, redémarrer une fois
    CLONE = Path("/content/pommes-case-studies")
    if not CLONE.exists():
        !git clone https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies {CLONE}
    %cd {CLONE}
    if not versions_du_depot():
        !python --version
        !pip install -q -r requirements.txt
        raise RedemarrezLaSession("Installation finie. Exécution › Redémarrer la session, "
                                  "puis relancez cette cellule (et seulement elle).")

RACINE = next((d for d in (Path.cwd(), *Path.cwd().parents, Path.cwd() / "pommes-case-studies")
               if (d / "pommes_cases").is_dir()), Path.cwd())  # la racine du dépôt des cas
if not (RACINE / "pommes_cases").is_dir():
    raise FileNotFoundError(f"pas de pommes_cases/ sous {RACINE} : placez ce notebook dans le dépôt "
                            "pommes-case-studies (à sa racine ou dans un de ses dossiers)")
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))
print("dépôt :", RACINE.name, "· Python", sys.version.split()[0])
print(" · ".join(f"{nom} {version(nom)}" for nom in VERSIONS))
if not versions_du_depot():
    print("Attention : ce ne sont pas les versions du dépôt", VERSIONS, ": les valeurs attendues peuvent bouger.")

Deux installations achètent l'électricité au prix horaire, pour fabriquer de l'hydrogène puis de la vapeur. Pommes choisit leur taille et leurs heures de marche ; vous retrouvez à la main ce qu'il décide. Première marche des cas D et E : un cas d'école, pas une prévision. Valeurs du 23/11/2026 (pommes 0.11.0, pommes_craft 0.6.0, HiGHS) ; blocs à exécuter à la suite, en ≈ 20 s puis ≈ 2 s.

## 1. Le squelette du premier modèle

On garde le squelette du premier modèle (`modele_de_base`) : série TYNDP 2026 NT+, France, 2030 (57,50 €/MWh en moyenne), une zone, une demande plate, un réseau qui vend à l'heure sans racheter, ni délestage ni écrêtement. Deux changements : la demande porte sur le produit, et chaque autre ressource reçoit son import, fermé (bornes à zéro) ou à prix fixe, sans quoi pommes l'ouvrirait gratuitement ; les années de sortie suivent les durées de vie (pièges 2 et 3 de [Les pièges connus de pommes et pommes_craft : symptôme, cause, remède](https://www.robingirard.eu/assets/parcours-es/latest/textes/pieges-pommes.html)).

In [ ]:
import sys, logging
import numpy as np
import polars as pl
from pommes_cases.prices import load_prices
from pommes_craft import (EnergyModel, Area, Demand, EconomicHypothesis, TimeStepManager, LoadShedding,
                          Spillage, NetImport, ConversionTechnology, StorageTechnology)
logging.getLogger().setLevel(logging.ERROR)
p = load_prices("tyndp2026", "NT+", "FR", 2030)
prix = p["import_price"].to_numpy()

def site(nom, produit, demande_mw, durees, achats={}):
    ressources = ["electricity", produit, *achats]
    m = EnergyModel(name=nom, hours=list(range(8760)), year_ops=[2030], year_invs=[2030],
                    year_decs=sorted({2030 + d for d in durees}), modes=[], resources=ressources,
                    solver_name="highs", solver_options={"output_flag": False})
    with m.context():
        EconomicHypothesis("eco", discount_rate=0.0, year_ref=2030, planning_step=1)
        TimeStepManager("pas", time_step_duration=1.0, operation_year_duration=8760.0)
        zone = Area("site")
        zone.add_component(Demand(name="demande", resource=produit,
                                  demand=pl.DataFrame({"hour": range(8760), "year_op": 2030, "demand": demande_mw})))
        for r in ressources:
            zone.add_component(LoadShedding(name=f"delestage_{r}", resource=r, max_capacity=0.0))
            zone.add_component(Spillage(name=f"ecretement_{r}", resource=r, max_capacity=0.0))
        zone.add_component(NetImport(name="reseau", resource="electricity", max_yearly_energy_export=0.0,
                                     import_price=p.select("hour", "year_op", "import_price")))
        zone.add_component(NetImport(name="ferme", resource=produit,
                                     max_yearly_energy_import=0.0, max_yearly_energy_export=0.0))
        for r, prix_fixe in achats.items():
            zone.add_component(NetImport(name=f"achat_{r}", resource=r, import_price=prix_fixe,
                                         max_yearly_energy_export=0.0))
    return m, zone

def marche(m, nom):
    return m.linopy_model.solution["operation_conversion_power"].sel(conversion_tech=nom).squeeze().values

## 2. Un électrolyseur face au prix

Le site consomme 100 MW d'hydrogène en continu (3 000 kg/h), fabriqués par un électrolyseur dont pommes choisit la taille. Pour sauter des heures chères, il faut un tampon qui serve le site pendant ses arrêts.

Un hydrogène de secours au coût variable d'un reformeur, qui le tire du gaz, coûte 69,29 € par MWh (2,31 €/kg), CO₂ compris au prix du scénario, 97,5 €/t : un grand site d'hydrogène relève du marché européen du carbone. Face à lui, un MW d'électrolyseur gagnerait au plus 91 697 €/an, pour 357 585 € d'annuité (remboursement annuel de l'investissement) et de coût fixe : pommes n'en construit aucun. Un réservoir sous pression, à 1 400 €/an par MWh de capacité, est trop cher : l'électrolyseur tourne alors 8 760 heures.

Reste le tampon le plus simple qui montre le mécanisme : une cavité saline, un stockage souterrain à 9,54 €/an par MWh de capacité, là où le sous-sol s'y prête. Coûts de 2040, le TYNDP ne l'ouvrant pas en 2030 ; canalisation jusqu'au site non comptée. Arguments du stockage : [Les composants de pommes_craft et leurs défauts](https://www.robingirard.eu/assets/parcours-es/latest/textes/fiche-composants-craft.html).

| cas D (`hypotheses.yaml`) | valeur |
|---|---|
| électrolyseur : investissement 2030 | 3 393 000 €/MW d'hydrogène |
| électricité consommée | 1,73 MWh par MWh d'hydrogène |
| coût fixe ; durée de vie | 12 000 €/MW/an ; 20 ans |
| cavité : investissement ; coût fixe | 80 €/MWh + 152 000 €/MW ; 3 €/MWh/an + 5 400 €/MW/an |
| cavité : pertes ; compression ; durée de vie | 2 % ; 0,0534 MWh électrique par MWh injecté ; 50 ans (100 dans le TYNDP) |

Statut : repris du TYNDP 2024, sauf la durée de vie de la cavité et le taux de 8 %, choix du cas.

In [ ]:
m, zone = site("electrolyseur", "hydrogen", 100.0, durees=[20, 50])
with m.context():
    zone.add_component(ConversionTechnology(
        name="electrolyseur", factor={"hydrogen": 1.0, "electricity": -1.73},
        invest_cost=3_393_000.0, fixed_cost=12_000.0, life_span=20, finance_rate=0.08))
    zone.add_component(StorageTechnology(
        name="cavite", factor_in={"hydrogen": -1.02, "electricity": -0.0534}, factor_out={"hydrogen": 1.0},
        factor_keep={"hydrogen": 0.0}, invest_cost_energy=80.0, invest_cost_power=152_000.0,
        fixed_cost_energy=3.0, fixed_cost_power=5_400.0, life_span=50, finance_rate=0.08))
m.run(store_linopy_model=True)
sol, P, kg = m.linopy_model.solution, marche(m, "electrolyseur"), 100 * 8760 * 30
print(f"{P.max():.1f} MW, cavité {float(sol['planning_storage_energy_capacity'].sum()):.0f} MWh")
# 160.0 MW, cavité 178530 MWh
print(f"{(P > 1e-6).sum()} heures, facteur de charge {P.mean() / P.max():.3f}")   # 5528 heures, facteur de charge 0.630
print(f"{(prix * P).sum() / P.sum():.2f} €/MWh, {m.linopy_model.objective.value / kg:.3f} €/kg")   # 36.18 €/MWh, 4.435 €/kg

L'électrolyseur fait 160 MW pour 100 MW de demande et tourne 63 % du temps (son facteur de charge, production moyenne rapportée à la taille). La cavité garde 178 530 MWh, 74 jours de demande, d'une saison à l'autre. L'électricité lui coûte 36,18 €/MWh, contre 57,50 pour un consommateur plat. Le kilogramme revient à 4,435 €, la valeur du cas D (`results.json`, `electrolyseur_cavite`), contre 2,47 € au reformeur (2,31 € de coût variable, 0,16 € de coût fixe pour l'unité gardée).

In [ ]:
plein, arret, n = P > P.max() - 1e-6, P < 1e-6, (P > 1e-6).sum()
print(plein.sum(), round(prix[plein].max(), 2), round(prix[arret].min(), 2), round(np.sort(prix)[:n].mean(), 2))
# 5494 76.11 76.7 36.26

def crf(i, n): return i * (1 + i) ** n / ((1 + i) ** n - 1)
def lcoh(capex_kwe, conso_mwh_t, duree, taux, fc, opex_frac, prix_elec):
    capex_t_an = capex_kwe * conso_mwh_t / 8.76
    return (capex_t_an * (crf(taux, duree) + opex_frac) / fc + conso_mwh_t * prix_elec) / 1000
print(round(lcoh(3_393_000 / 1.73 / 1000, 1.73 * 1000 / 30, 20, 0.08, 0.630, 12_000 / 3_393_000, 36.18), 3))   # 4.246
print(round(float(sol["planning_storage_costs"].sum() + sol["operation_storage_costs"].sum()) / kg, 3))  # 0.133

Le seuil est presque net : 5 494 heures à plein jusqu'à 76,11 €/MWh, arrêt dès 76,70 ; les 34 autres heures, entre 71,5 et 80 €/MWh, il produit tout juste les 100 MW de la demande, cavité au repos. Marchant presque toujours à plein ou à l'arrêt, sa part d'heures de marche (5 528 sur 8 760, 0,631) rejoint son facteur de charge (0,630). Tenu de fabriquer l'hydrogène de l'année, il prend les heures les moins chères qui suffisent : son prix payé approche la moyenne des 5 528 moins chères (36,26). C'est la monotone (les prix triés) de « Lire une série de prix » (3 000 heures à 22,4 €/MWh), quantité imposée en plus.

Pourquoi 160 MW ? Un MW de plus fabriquerait davantage aux heures retenues, vers 36 €/MWh, et autant de moins aux heures les plus chères encore achetées, vers 76 €/MWh ; cavité, compression et pertes payées, cette économie égale tout juste son coût, 357 585 €/an. Imposer 161 MW coûte 14 525 € de plus.

Avec ce facteur de charge et ce prix payé, la fonction `lcoh` du texte « deux chiffres » donne 4,246 €/kg pour l'électrolyseur seul ; les 0,19 €/kg manquants paient le tampon : 0,133 la cavité, 0,056 sa compression et ses 2 % de pertes.

## 3. Deux chaudières face au prix

Le site consomme 1 MW de vapeur en continu ; pommes choisit la taille de ses deux chaudières. L'électrique paie l'électricité à l'heure, l'autre le gaz à prix fixe et son CO₂. Une chaudière de 1 MW relève de l'ETS2, le marché du carbone des petits sites : on prend son seuil de 45 €/t (euros de 2020), au-delà duquel sa réserve libère des quotas, ni une cible, ni un plafond, ni une prévision.

| cas E (`hypotheses.yaml`) | chaudière au gaz | chaudière électrique |
|---|---|---|
| rendement | 0,94 | 0,99 |
| investissement | 90 000 €/MW | 220 000 €/MW |
| coût fixe ; durée de vie | 2 000 €/MW/an ; 25 ans | 1 375 €/MW/an ; 25 ans |
| énergie | 33,2 €/MWh + 45 €/t × 0,202 t/MWh | prix horaire |

Statut : vérifié (DEA, fiches 6.1 et 5.1a, 2030 ; gaz du TYNDP 2026 ; 0,202 t/MWh du GIEC), sauf 45 €/t et le taux de 8 %, choix du cas.

À la main d'abord. Un MWh de vapeur coûte (33,2 + 0,202 × 45) / 0,94 = 44,99 € au gaz, prix / 0,99 à l'électricité. Les deux s'égalent au prix de bascule, 44,99 × 0,99 = 44,54 €/MWh : en dessous, l'électrique coûte moins cher à faire tourner.

In [ ]:
gaz = 33.2 + 0.202 * 45
seuil = gaz / 0.94 * 0.99
print(round(seuil, 2), (prix < seuil).sum())   # 44.54 3680
m, zone = site("chaudieres", "vapeur", 1.0, durees=[25], achats={"methane": gaz})
with m.context():
    zone.add_component(ConversionTechnology(name="chaudiere_gaz", factor={"vapeur": 1.0, "methane": -1 / 0.94},
                                            invest_cost=90_000.0, fixed_cost=2_000.0, life_span=25, finance_rate=0.08))
    zone.add_component(ConversionTechnology(name="chaudiere_elec", factor={"vapeur": 1.0, "electricity": -1 / 0.99},
                                            invest_cost=220_000.0, fixed_cost=1_375.0, life_span=25, finance_rate=0.08))
m.run(store_linopy_model=True)
E, G = marche(m, "chaudiere_elec"), marche(m, "chaudiere_gaz")
on = E > 1e-6
print(E.max(), G.max(), on.sum(), (on == (prix < seuil)).sum())   # 1.0 1.0 3680 8760
print(round(prix[on].max(), 2), round(prix[~on].min(), 2), round(E.sum() / 8760, 3))   # 44.31 44.6 0.42
print(round(m.linopy_model.objective.value / 8760, 2))   # 40.79

Pommes retrouve la bascule heure par heure : l'électrique tourne les 3 680 heures sous 44,54 €/MWh et aucune autre (accord sur 8 760 heures), soit 42,0 % de la vapeur. Chaque chaudière fait 1 MW : avec une demande plate, chaque MW électrique économise la même somme, pommes en prend tout ou rien, et le gaz couvre les 5 080 heures chères. La vapeur revient à 40,79 €/MWh.

Le cas E ajoute deux postes de 8 h, un stockage de chaleur, plusieurs jeux de prix et de CO₂, et en variante le tarif de réseau (puissance souscrite comprise) et les taxes.

## 4. Une troisième installation

La fiche des composants (section « Activité ») ajoute au premier modèle une batterie de 1 MW / 2 h : elle achète aussi à l'heure, pour restituer plus tard à la zone, à la place d'un achat plus cher ou de la turbine.

## Vous l'avez compris quand…

- Vous retrouvez 160,0 MW, un facteur de charge de 0,630, 36,18 €/MWh payés contre 57,50 et 4,435 €/kg, et expliquez pourquoi ce prix approche la moyenne des 5 528 heures les moins chères (36,26).
- Vous remplacez la cavité par les réservoirs du cas D (10 700 €/MWh, 69 000 €/MW, 450 €/MWh/an, 2 930 €/MW/an, compression 0,0512, 30 ans ; `durees=[20, 30]`, sans quoi le réservoir ne peut exister) et prédisez : aucun réservoir, 100 MW, 8 760 heures, 4,677 €/kg.
- Vous recalculez la bascule (44,54 €/MWh) et retrouvez dans la solution ses 3 680 heures et la part de 42,0 %.
- Vous ajoutez les coûts variables d'entretien de la DEA, laissés de côté ici (`variable_cost=1.23` au gaz, `0.642` à l'électrique), et prédisez le seuil, (44,99 + 1,23 − 0,642) × 0,99 = 45,12 €/MWh : 3 724 heures électriques, 44 de plus, 42,5 % de la vapeur.

## Pour aller plus loin

- Le premier modèle [Le premier modèle sur une page : les huit étapes du notebook et leurs valeurs attendues](https://www.robingirard.eu/assets/parcours-es/latest/textes/premier-modele.html).
- Les cas D [Cas D — Électrolyseur : hybridation, stockage, et que coûte la corrélation horaire de 2030 ?](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies/-/tree/main/D_electrolyseur) et E [Cas E — Vapeur à 180 °C : tout électrifier, garder le gaz, ou les deux ?](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies/-/tree/main/E_vapeur) ; 4,8 ou 7,1 €/kg pour le même hydrogène [Pourquoi deux chiffres pour le même hydrogène : cas D (4,4 à 4,9 €/kg) et outil « routes » (7,1 €/kg)](https://www.robingirard.eu/assets/parcours-es/latest/textes/hydrogene-deux-chiffres.html).
- La monotone et les heures bon marché [Lire une série de prix horaires : six choses à regarder et les chiffres du cas 0](https://www.robingirard.eu/assets/parcours-es/latest/textes/lire-un-prix.html).

## Sources

1. [Dépôt pommes-case-studies (D_electrolyseur/hypotheses.yaml et results.json, E_vapeur/hypotheses.yaml et results.json, pommes_cases/blocks.py)](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies)
2. [ENTSO-E et ENTSOG, TYNDP 2024 Scenarios, téléchargements (coûts de l'électrolyseur et des stockages d'hydrogène)](https://2024.entsos-tyndp-scenarios.eu/download/)
3. [ENTSO-E et ENTSOG, TYNDP 2026 Scenarios, téléchargements (prix horaires, gaz et CO₂ de 2030)](https://2026.entsos-tyndp-scenarios.eu/download/)
4. [Danish Energy Agency, Technology Data for Industrial Process Heat (fiches 5.1a et 6.1)](https://ens.dk/en/analyses-and-statistics/technology-data-industrial-process-heat)
5. [Directive (UE) 2023/959, art. 30 nonies : le seuil de 45 €/t de l'ETS2](https://eur-lex.europa.eu/eli/dir/2023/959/oj)
6. [Usage de pommes_craft](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes_craft/usage.html)